In [1]:
from PIL import Image
import plotly.graph_objects as go
import pandas as pd
import plotly.io as pio
import plotly.express as px
from pyproj import Transformer
from sklearn.neighbors import NearestNeighbors
import numpy as np

In [2]:
#Load the data and img
pio.renderers.default = "notebook"
img = Image.open("snowmap.png") 
deathData=pd.read_csv("deaths.csv")
pumpData=pd.read_csv("pumps.csv")
snowDates=pd.read_csv("snow_dates.csv")
pumpData

,pump_name,x_coordinate,y_coordinate
0,Broad St.,51.513341,-0.136668
1,Crown Chapel,51.513876,-0.139586
2,Gt Marlborough,51.514906,-0.139671
3,Dean St.,51.512354,-0.131630
4,So Soho,51.512139,-0.133594
5,Briddle St.,51.511542,-0.135919
6,Coventry St.,51.510019,-0.133962
7,Warwick,51.511295,-0.138199


In [3]:
#Standardize latitude and longitude
transformer = Transformer.from_crs("EPSG:4326", "EPSG:32630", always_xy=True)

broad = pumpData.loc[pumpData["pump_name"] == "Broad St."].iloc[0]
origin_x, origin_y = transformer.transform(broad["y_coordinate"], broad["x_coordinate"])

def convert_meters(lon, lat):
    x, y = transformer.transform(lon.to_numpy(), lat.to_numpy())
    return (x - origin_x), (y - origin_y)

deathData["x_meters"], deathData["y_meters"] = convert_meters(deathData["y_longitude"], deathData["x_latitude"])
pumpData["x_meters"],  pumpData["y_meters"]  = convert_meters(pumpData["y_coordinate"], pumpData["x_coordinate"])

In [4]:
deathData.head()

,death_count,x_latitude,y_longitude,x_meters,y_meters
0,1,51.513418,-0.137930,-87.883721,5.134228
1,1,51.513418,-0.137930,-87.883721,5.134228
2,1,51.513418,-0.137930,-87.883721,5.134228
3,1,51.513361,-0.137883,-84.375190,-1.075571
4,1,51.513361,-0.137883,-84.375190,-1.075571


In [5]:
pumpData.head()

,pump_name,x_coordinate,y_coordinate,x_meters,y_meters
0,Broad St.,51.513341,-0.136668,0.000000,0.000000
1,Crown Chapel,51.513876,-0.139586,-204.756789,51.561388
2,Gt Marlborough,51.514906,-0.139671,-215.132207,165.848653
3,Dean St.,51.512354,-0.131630,353.804643,-96.041672
4,So Soho,51.512139,-0.133594,218.490139,-125.287652


In [6]:
#Combine deaths in same location 
deaths_grouped = (deathData.groupby(["x_meters", "y_meters", "x_latitude", "y_longitude"],
                                    as_index=False)["death_count"].sum())
deaths_grouped.head()

,x_meters,y_meters,x_latitude,y_longitude,death_count
0,-236.097469,-14.025973,51.513298,-0.140074,1
1,-212.526638,13.954436,51.513541,-0.139719,2
2,-208.704576,-44.690181,51.513013,-0.139697,1
3,-205.563881,18.903549,51.513583,-0.139616,2
4,-185.446857,28.598569,51.513663,-0.139321,1


In [9]:
fig = go.Figure()

fig.add_trace(go.Scattermap(
    lat=deaths_grouped["x_latitude"], lon=deaths_grouped["y_longitude"],
    mode="markers", name="Deaths",
    text=deaths_ll["death_count"],
    hovertemplate="<b>Deaths at this location:</b> %{text}<extra></extra>",
    marker=dict(size=6 + deaths_ll["death_count"] * 2, color="crimson", opacity=0.6)
))

fig.add_trace(go.Scattermap(
    lat=pumpData["x_coordinate"], lon=pumpData["y_coordinate"],
    mode="markers+text", name="Water Pumps",
    text=pumpData["pump_name"], textposition="top center",
    marker=dict(size=12, color="dodgerblue")
))

fig.update_layout(
    map=dict(style="carto-positron",
             center=dict(lat=51.5133, lon=-0.1367), zoom=15.5),
    width=1000, height=700
)
fig.show()